# Saree Virtual Try-On — Full Training Pipeline
**Model:** VITON-HD style (SareeWarpingNetwork + TryOnGenerator)  
**GPU:** Enable T4/P100 in Kaggle Settings → Accelerator → GPU  

### Steps
1. Install dependencies
2. Configure Kaggle API & download datasets
3. Clean & preprocess dataset
4. Extract pose maps + segmentation masks
5. Define model architecture
6. Train (GMM warp + TOM generator)
7. Validate & visualise
8. Export weights → download

## 1 · Install Dependencies

In [ ]:
%%capture
!pip install -q kaggle mediapipe albumentations kornia tqdm scikit-image
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu118

import torch, sys
print('Python :', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('CUDA   :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU    :', torch.cuda.get_device_name(0))
    print('VRAM   :', round(torch.cuda.get_device_properties(0).total_memory/1e9,1),'GB')

## 2 · Kaggle API & Dataset Download

In [ ]:
import os, json

# ── Paste your Kaggle credentials (Settings → API → Create New Token) ──────────
KAGGLE_CREDS = {
    'username': 'YOUR_KAGGLE_USERNAME',   # <-- replace
    'key':      'YOUR_KAGGLE_API_KEY'     # <-- replace
}

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json','w') as f:
    json.dump(KAGGLE_CREDS, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)

# ── Datasets (try each; skip if unavailable) ───────────────────────────────────
DATASETS = [
    ('manishkumar2311/indian-traditional-women-saree-fashion', 'saree_main'),
    ('balabaskar/human-images-dataset',                        'human_imgs'),
    ('tapakah68/upper-body-clothes',                           'upper_body'),
]

RAW = '/kaggle/working/raw'
os.makedirs(RAW, exist_ok=True)

for slug, name in DATASETS:
    dest = f'{RAW}/{name}'
    if os.path.exists(dest) and os.listdir(dest):
        print(f'  skip (exists): {name}')
        continue
    print(f'Downloading {slug}...')
    r = os.system(f'kaggle datasets download -d {slug} -p {dest} --unzip -q 2>/dev/null')
    n = sum(len(fs) for _,_,fs in os.walk(dest))
    print(f'  {"✓" if r==0 else "✗"} {name}: {n} files')

## 3 · Dataset Cleaning & Preprocessing

In [ ]:
import cv2, numpy as np
from pathlib import Path
from PIL import Image
from tqdm import tqdm

IMG_H, IMG_W = 256, 192     # Model input resolution
MIN_DIM      = 150           # Reject images smaller than this

CLEAN = Path('/kaggle/working/clean')
(CLEAN/'person').mkdir(parents=True, exist_ok=True)
(CLEAN/'saree').mkdir(parents=True, exist_ok=True)
(CLEAN/'blouse').mkdir(parents=True, exist_ok=True)


def is_valid(p):
    try:
        img = cv2.imread(str(p))
        if img is None: return False
        h,w = img.shape[:2]
        if min(h,w) < MIN_DIM: return False
        if cv2.cvtColor(img, cv2.COLOR_BGR2GRAY).std() < 8: return False
        return True
    except: return False


def letterbox_save(src, dst, size=(IMG_W, IMG_H)):
    try:
        img = Image.open(src).convert('RGB')
        w,h = img.size
        s   = min(size[0]/w, size[1]/h)
        nw,nh = int(w*s), int(h*s)
        img   = img.resize((nw,nh), Image.LANCZOS)
        canvas = Image.new('RGB', size, (255,255,255))
        canvas.paste(img, ((size[0]-nw)//2, (size[1]-nh)//2))
        canvas.save(dst, quality=95)
        return True
    except: return False


def has_skin(bgr):
    hsv  = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv,
        np.array([0,20,70],  dtype=np.uint8),
        np.array([25,255,255],dtype=np.uint8))
    h,w = bgr.shape[:2]
    return mask.sum()/(h*w*255) > 0.025


all_imgs = list(Path(RAW).rglob('*.jpg'))  + \
           list(Path(RAW).rglob('*.jpeg')) + \
           list(Path(RAW).rglob('*.png'))
# also check /kaggle/input if user added datasets via UI
if Path('/kaggle/input').exists():
    all_imgs += list(Path('/kaggle/input').rglob('*.jpg')) + \
                list(Path('/kaggle/input').rglob('*.jpeg'))

print(f'Total raw images: {len(all_imgs)}')

person_paths, saree_paths, blouse_paths = [], [], []

for p in tqdm(all_imgs, desc='Classifying'):
    if not is_valid(p): continue
    bgr = cv2.imread(str(p))
    h,w = bgr.shape[:2]
    aspect = h/w
    skin   = has_skin(bgr)
    pname  = str(p).lower()
    # Heuristic classification
    if skin and aspect > 1.3:
        person_paths.append(p)
    elif 'blouse' in pname or ('top' in pname and not skin):
        blouse_paths.append(p)
    elif not skin and aspect > 0.8:
        saree_paths.append(p)

print(f'Person: {len(person_paths)} | Saree: {len(saree_paths)} | Blouse: {len(blouse_paths)}')

In [ ]:
# Save cleaned images
MAX_PER_CLASS = 4000
saved = {'person':[], 'saree':[], 'blouse':[]}

for cls, paths in [('person',person_paths),('saree',saree_paths),('blouse',blouse_paths)]:
    for i, p in enumerate(tqdm(paths[:MAX_PER_CLASS], desc=f'Saving {cls}')):
        dst = str(CLEAN/cls/f'{cls}_{i:05d}.jpg')
        if letterbox_save(p, dst):
            saved[cls].append(dst)

N = min(len(saved['person']), len(saved['saree']), len(saved['blouse']))
pairs = [{'person': saved['person'][i],
           'saree':  saved['saree'][i],
           'blouse': saved['blouse'][i % len(saved['blouse'])]} for i in range(N)]

with open(CLEAN/'pairs.json','w') as f: json.dump(pairs, f)
print(f'Dataset ready: {N} triplets (person, saree, blouse)')

In [ ]:
# Visualise samples
import matplotlib.pyplot as plt
fig, axes = plt.subplots(3, 6, figsize=(18,10))
for row, cls in enumerate(['person','saree','blouse']):
    for col, p in enumerate(saved[cls][:6]):
        axes[row,col].imshow(Image.open(p))
        axes[row,col].set_title(f'{cls} {col}', fontsize=8)
        axes[row,col].axis('off')
plt.suptitle('Cleaned Dataset Samples', fontsize=13)
plt.tight_layout()
plt.savefig('/kaggle/working/samples.png', dpi=90)
plt.show()

## 4 · Pose Maps & Segmentation Masks

In [ ]:
import mediapipe as mp, urllib.request

MODEL_FILE = '/kaggle/working/pose_landmarker_full.task'
if not os.path.exists(MODEL_FILE):
    urllib.request.urlretrieve(
        'https://storage.googleapis.com/mediapipe-models/pose_landmarker/'
        'pose_landmarker_full/float16/latest/pose_landmarker_full.task',
        MODEL_FILE)
    print('Pose model downloaded')

from mediapipe.tasks import python as mp_tasks
from mediapipe.tasks.python import vision as mp_vision
from mediapipe.tasks.python.vision.core.vision_task_running_mode import VisionTaskRunningMode

POSE_DIR = CLEAN / 'pose'
POSE_DIR.mkdir(exist_ok=True)

KP_NAMES = ['nose','left_eye_inner','left_eye','left_eye_outer',
             'right_eye_inner','right_eye','right_eye_outer','left_ear','right_ear',
             'mouth_left','mouth_right','left_shoulder','right_shoulder',
             'left_elbow','right_elbow','left_wrist','right_wrist',
             'left_pinky','right_pinky','left_index','right_index',
             'left_thumb','right_thumb','left_hip','right_hip',
             'left_knee','right_knee','left_ankle','right_ankle',
             'left_heel','right_heel','left_foot_index','right_foot_index']

pose_opts = mp_vision.PoseLandmarkerOptions(
    base_options=mp_tasks.BaseOptions(model_asset_path=MODEL_FILE),
    running_mode=VisionTaskRunningMode.IMAGE,
    num_poses=1, min_pose_detection_confidence=0.3)

pose_index = {}
no_pose    = 0

with mp_vision.PoseLandmarker.create_from_options(pose_opts) as lm:
    for p in tqdm(saved['person'], desc='Pose extraction'):
        bgr = cv2.imread(p)
        if bgr is None: continue
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        res = lm.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb))
        if not res.pose_landmarks:
            no_pose += 1; continue
        h, w = bgr.shape[:2]
        kp = {KP_NAMES[i]: {'x': round(lm_.x*w,2), 'y': round(lm_.y*h,2),
                              'visibility': round(getattr(lm_,'visibility',1.0),3)}
              for i, lm_ in enumerate(res.pose_landmarks[0]) if i < len(KP_NAMES)}
        dst = str(POSE_DIR / (Path(p).stem + '_pose.json'))
        with open(dst,'w') as f: json.dump({'keypoints':kp,'image_size':{'w':w,'h':h}}, f)
        pose_index[p] = dst

with open(CLEAN/'pose_index.json','w') as f: json.dump(pose_index, f)
print(f'Poses: {len(pose_index)} OK, {no_pose} failed')

In [ ]:
# Generate segmentation masks for person images (GrabCut fallback)
SEG_DIR = CLEAN / 'seg'
SEG_DIR.mkdir(exist_ok=True)

def grabcut_mask(path, h=IMG_H, w=IMG_W):
    bgr  = cv2.imread(path)
    bgr  = cv2.resize(bgr, (w, h))
    mask = np.zeros((h,w), np.uint8)
    bgd  = np.zeros((1,65), np.float64)
    fgd  = np.zeros((1,65), np.float64)
    rect = (w//10, h//15, w - 2*(w//10), h - 2*(h//15))
    cv2.grabCut(bgr, mask, rect, bgd, fgd, 5, cv2.GC_INIT_WITH_RECT)
    fg = np.where((mask==2)|(mask==0), 0, 255).astype(np.uint8)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(7,7))
    fg = cv2.morphologyEx(fg, cv2.MORPH_CLOSE, kernel)
    return (fg > 127).astype(np.uint8) * 255

seg_index = {}
for p in tqdm(saved['person'][:len(pose_index)], desc='Segmentation'):
    try:
        mask = grabcut_mask(p)
        dst  = str(SEG_DIR / (Path(p).stem + '_seg.png'))
        cv2.imwrite(dst, mask)
        seg_index[p] = dst
    except: pass

with open(CLEAN/'seg_index.json','w') as f: json.dump(seg_index, f)
print(f'Seg masks: {len(seg_index)}')

## 5 · Model Architecture

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional, Tuple

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Reuse the exact same classes as backend/ai_engine/ml_models.py ────────────
# (Inline here so the notebook is self-contained)

class ResBlock(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.ReflectionPad2d(1), nn.Conv2d(ch,ch,3,bias=False),
            nn.InstanceNorm2d(ch), nn.ReLU(inplace=True),
            nn.ReflectionPad2d(1), nn.Conv2d(ch,ch,3,bias=False),
            nn.InstanceNorm2d(ch))
    def forward(self,x): return x + self.block(x)

class DownBlock(nn.Module):
    def __init__(self, ic, oc, norm=True, drop=0.0):
        super().__init__()
        L = [nn.Conv2d(ic,oc,4,2,1,bias=not norm)]
        if norm: L.append(nn.InstanceNorm2d(oc))
        L.append(nn.LeakyReLU(0.2,inplace=True))
        if drop: L.append(nn.Dropout(drop))
        self.block = nn.Sequential(*L)
    def forward(self,x): return self.block(x)

class UpBlock(nn.Module):
    def __init__(self, ic, oc, drop=0.0):
        super().__init__()
        L = [nn.ConvTranspose2d(ic,oc,4,2,1,bias=False),
             nn.InstanceNorm2d(oc), nn.ReLU(inplace=True)]
        if drop: L.append(nn.Dropout(drop))
        self.block = nn.Sequential(*L)
    def forward(self,x,skip): return self.block(torch.cat([x,skip],1))

class FeatureEncoder(nn.Module):
    def __init__(self, in_ch=3, base_ch=64):
        super().__init__()
        self.e1 = nn.Sequential(nn.Conv2d(in_ch,base_ch,7,1,3,bias=False),
                                  nn.InstanceNorm2d(base_ch), nn.ReLU(inplace=True))
        self.e2 = DownBlock(base_ch,   base_ch*2)
        self.e3 = DownBlock(base_ch*2, base_ch*4)
        self.e4 = DownBlock(base_ch*4, base_ch*8)
        self.e5 = DownBlock(base_ch*8, base_ch*8)
        self.res= nn.Sequential(*[ResBlock(base_ch*8) for _ in range(4)])
    def forward(self,x):
        f1=self.e1(x); f2=self.e2(f1); f3=self.e3(f2)
        f4=self.e4(f3); f5=self.res(self.e5(f4))
        return [f1,f2,f3,f4,f5]

class SareeWarpingNetwork(nn.Module):
    def __init__(self, gs=5, H=256, W=192, base_ch=64):
        super().__init__()
        self.gs=gs; self.H=H; self.W=W
        self.pe = FeatureEncoder(3+18, base_ch)
        self.ge = FeatureEncoder(3,    base_ch)
        fh,fw   = H//32, W//32
        corr_d  = fh*fw*fh*fw
        self.tr = nn.Sequential(
            nn.Flatten(), nn.Linear(corr_d,512), nn.ReLU(),
            nn.Dropout(0.3), nn.Linear(512,256), nn.ReLU(),
            nn.Linear(256, gs*gs*2), nn.Tanh())
    def _corr(self,fp,fg):
        b,c,h,w=fp.shape
        fp=F.normalize(fp.view(b,c,-1),dim=1)
        fg=F.normalize(fg.view(b,c,-1),dim=1)
        return F.relu(torch.bmm(fp.permute(0,2,1),fg)).view(b,h*w,h,w)
    def _grid(self,theta,b,dev):
        g=self.gs
        xs=torch.linspace(-1,1,g,device=dev)
        ys=torch.linspace(-1,1,g,device=dev)
        gx,gy=torch.meshgrid(xs,ys,indexing='xy')
        base=torch.stack([gx.reshape(-1),gy.reshape(-1)],dim=-1).unsqueeze(0).expand(b,-1,-1)
        ctrl=(base+theta.view(b,g*g,2).clamp(-0.5,0.5)).view(b,g,g,2).permute(0,3,1,2)
        grid=F.interpolate(ctrl,size=(self.H,self.W),mode='bilinear',align_corners=True)
        return grid.permute(0,2,3,1)
    def forward(self,pp,g,seg=None):
        fp=self.pe(pp)[-1]; fg=self.ge(g)[-1]
        if seg is not None:
            s=F.adaptive_avg_pool2d(seg.float(),fp.shape[-2:]); fp=fp*(0.5+0.5*s)
        c=self._corr(fp,fg); t=self.tr(c).view(g.size(0),-1,2)
        grid=self._grid(t,g.size(0),g.device)
        return F.grid_sample(g,grid,mode='bilinear',padding_mode='border',align_corners=True), grid

class TryOnGenerator(nn.Module):
    def __init__(self, in_ch=29, base_ch=64):
        super().__init__()
        self.d1=DownBlock(in_ch,     base_ch,  norm=False)
        self.d2=DownBlock(base_ch,   base_ch*2)
        self.d3=DownBlock(base_ch*2, base_ch*4)
        self.d4=DownBlock(base_ch*4, base_ch*8, drop=0.4)
        self.d5=DownBlock(base_ch*8, base_ch*8, drop=0.4)
        self.d6=DownBlock(base_ch*8, base_ch*8, drop=0.4)
        self.d7=DownBlock(base_ch*8, base_ch*8, norm=False, drop=0.4)
        self.res=nn.Sequential(*[ResBlock(base_ch*8) for _ in range(3)])
        self.u1=UpBlock(base_ch*8,   base_ch*8, drop=0.4)
        self.u2=UpBlock(base_ch*16,  base_ch*8, drop=0.4)
        self.u3=UpBlock(base_ch*16,  base_ch*8, drop=0.4)
        self.u4=UpBlock(base_ch*16,  base_ch*4)
        self.u5=UpBlock(base_ch*8,   base_ch*2)
        self.u6=UpBlock(base_ch*4,   base_ch)
        self.out=nn.Sequential(nn.ConvTranspose2d(base_ch*2,4,4,2,1), nn.Tanh())
    def forward(self,x):
        d1=self.d1(x);d2=self.d2(d1);d3=self.d3(d2);d4=self.d4(d3)
        d5=self.d5(d4);d6=self.d6(d5);d7=self.res(self.d7(d6))
        u=self.u6(self.u5(self.u4(self.u3(self.u2(self.u1(d7,d6),d5),d4),d3),d2),d1)
        o=self.out(torch.cat([u,d1],1)); rgb=o[:,:3]; a=(o[:,3:4]+1)/2
        return a*rgb+(1-a)*x[:,:3], a

class SareeTryOnModel(nn.Module):
    FABRIC_TYPES   = ['silk','cotton','georgette','chiffon','other']
    DRAPING_STYLES = ['nivi','bridal','hanging','gujarati']
    def __init__(self, H=256, W=192, gs=5, base_ch=64):
        super().__init__()
        self.style_emb  = nn.Embedding(4, 64)
        self.style_proj = nn.Sequential(nn.Linear(64, H*W), nn.Tanh())
        self.sw  = SareeWarpingNetwork(gs,H,W,base_ch)
        self.bw  = SareeWarpingNetwork(gs,H,W,base_ch//2)
        self.gen = TryOnGenerator(29, base_ch)
    def forward(self,person,saree,blouse,pose_heatmap,seg_mask,fabric_idx=None,style_idx=None):
        b,_,h,w=person.shape
        s_map=(self.style_proj(self.style_emb(style_idx)).view(b,1,h,w)
               if style_idx is not None else torch.zeros(b,1,h,w,device=person.device))
        pp=torch.cat([person,pose_heatmap],1)
        ws,sg=self.sw(pp,saree,seg_mask)
        wb,bg=self.bw(pp,blouse,seg_mask)
        inp=torch.cat([person,ws,wb,pose_heatmap,seg_mask,s_map],1)
        rendered,alpha=self.gen(inp)
        return {'warped_saree':ws,'warped_blouse':wb,'rendered':rendered,'alpha_mask':alpha,'saree_grid':sg,'blouse_grid':bg}

model=SareeTryOnModel(H=IMG_H,W=IMG_W).to(DEVICE)
p=sum(x.numel() for x in model.parameters())/1e6
print(f'Model ready: {p:.1f}M params on {DEVICE}')

## 6 · Dataset Loader

In [ ]:
import torchvision.transforms as T
import albumentations as A
from torch.utils.data import Dataset, DataLoader
import random

HEATMAP_KPS = ['left_shoulder','right_shoulder','left_elbow','right_elbow',
                'left_wrist','right_wrist','left_hip','right_hip',
                'left_knee','right_knee','left_ankle','right_ankle',
                'nose','left_ear','right_ear','left_eye','right_eye','mouth_left']

def make_heatmap(kp, h, w, sigma=8.0):
    out=np.zeros((18,h,w),dtype=np.float32)
    xs,ys=np.mgrid[0:w,0:h].astype(np.float32)
    for i,name in enumerate(HEATMAP_KPS):
        if name not in kp: continue
        v=kp[name]; vis=v.get('visibility',1.0)
        if vis<0.15: continue
        out[i]=np.exp(-((xs-v['x'])**2+(ys-v['y'])**2)/(2*sigma**2)).T
    return out

NORM = T.Normalize([0.5]*3,[0.5]*3)
TO_T = T.ToTensor()
AUG  = A.Compose([A.HorizontalFlip(p=0.5),
                   A.RandomBrightnessContrast(p=0.3),
                   A.HueSaturationValue(10,15,10,p=0.3),
                   A.GaussNoise(var_limit=(5,15),p=0.2)])

class SareeDS(Dataset):
    def __init__(self, pairs, pose_idx, seg_idx, augment=True):
        # Only keep pairs where person has both pose and seg
        self.pairs=[p for p in pairs if p['person'] in pose_idx and p['person'] in seg_idx]
        self.pose_idx=pose_idx; self.seg_idx=seg_idx; self.aug=augment
    def __len__(self): return len(self.pairs)
    def _load(self, path):
        img=np.array(Image.open(path).convert('RGB').resize((IMG_W,IMG_H),Image.LANCZOS))
        if self.aug: img=AUG(image=img)['image']
        return NORM(TO_T(img))
    def __getitem__(self,i):
        p=self.pairs[i]
        person=self._load(p['person']); saree=self._load(p['saree']); blouse=self._load(p['blouse'])
        with open(self.pose_idx[p['person']]) as f: pd=json.load(f)
        kp=pd['keypoints']; ow,oh=pd['image_size']['w'],pd['image_size']['h']
        kps={n:{'x':v['x']*IMG_W/ow,'y':v['y']*IMG_H/oh,'visibility':v.get('visibility',1.0)} for n,v in kp.items()}
        hm=torch.from_numpy(make_heatmap(kps,IMG_H,IMG_W))
        seg_raw=cv2.imread(self.seg_idx[p['person']],cv2.IMREAD_GRAYSCALE)
        seg=torch.from_numpy(cv2.resize(seg_raw,(IMG_W,IMG_H)).astype(np.float32)/255.0).unsqueeze(0)
        style_idx=torch.tensor(random.randint(0,3),dtype=torch.long)
        fabric_idx=torch.tensor(random.randint(0,4),dtype=torch.long)
        return dict(person=person,saree=saree,blouse=blouse,
                    pose=hm,seg=seg,style_idx=style_idx,fabric_idx=fabric_idx)

with open(CLEAN/'pairs.json') as f:      pairs    =json.load(f)
with open(CLEAN/'pose_index.json') as f: pose_idx =json.load(f)
with open(CLEAN/'seg_index.json') as f:  seg_idx  =json.load(f)

random.shuffle(pairs)
nv=max(1,int(len(pairs)*0.1))
tr_ds=SareeDS(pairs[nv:],pose_idx,seg_idx,augment=True)
vl_ds=SareeDS(pairs[:nv],pose_idx,seg_idx,augment=False)

BS=8
tr_dl=DataLoader(tr_ds,BS,shuffle=True, num_workers=2,pin_memory=True,drop_last=True)
vl_dl=DataLoader(vl_ds,BS,shuffle=False,num_workers=2,pin_memory=True)
print(f'Train: {len(tr_ds)} ({len(tr_dl)} batches) | Val: {len(vl_ds)} ({len(vl_dl)} batches)')

## 7 · Training

In [ ]:
import torchvision.models as tvm

class VGGLoss(nn.Module):
    def __init__(self):
        super().__init__()
        try: vgg=tvm.vgg16(weights=tvm.VGG16_Weights.DEFAULT)
        except: vgg=tvm.vgg16(pretrained=False)
        self.feat=nn.Sequential(*list(vgg.features)[:16]).eval()
        for p in self.parameters(): p.requires_grad_(False)
    def forward(self,p,t):
        return F.l1_loss(self.feat(p),self.feat(t))

def grid_reg(grid):
    dx=grid[:,:,1:,:]-grid[:,:,:-1,:]
    dy=grid[:,1:,:,:]-grid[:,:-1,:,:]
    return (dx.pow(2).mean()+dy.pow(2).mean())*0.5

vgg_loss=VGGLoss().to(DEVICE)
opt=torch.optim.Adam(model.parameters(),lr=2e-4,betas=(0.5,0.999))
sch=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=30,eta_min=5e-6)

CKPT_DIR=Path('/kaggle/working/checkpoints'); CKPT_DIR.mkdir(exist_ok=True)
N_EPOCHS=30
history={'train':[],'val':[]}
best_val=float('inf')

for epoch in range(N_EPOCHS):
    # ── Train ────────────────────────────────────────────────────────────────
    model.train(); t_losses=[]
    for b in tqdm(tr_dl, desc=f'Ep {epoch+1}/{N_EPOCHS} [train]', leave=False):
        b={k:v.to(DEVICE) for k,v in b.items()}
        opt.zero_grad()
        out=model(b['person'],b['saree'],b['blouse'],b['pose'],b['seg'],b['fabric_idx'],b['style_idx'])
        l1   = F.l1_loss(out['rendered'], b['person'])
        perc = vgg_loss((out['rendered']+1)/2, (b['person']+1)/2)
        reg  = grid_reg(out['saree_grid']) + grid_reg(out['blouse_grid'])
        loss = l1 + 0.2*perc + 0.01*reg
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
        opt.step()
        t_losses.append(loss.item())
    # ── Validate ─────────────────────────────────────────────────────────────
    model.eval(); v_losses=[]
    with torch.no_grad():
        for b in vl_dl:
            b={k:v.to(DEVICE) for k,v in b.items()}
            out=model(b['person'],b['saree'],b['blouse'],b['pose'],b['seg'],b['fabric_idx'],b['style_idx'])
            v_losses.append(F.l1_loss(out['rendered'],b['person']).item())
    tm,vm=np.mean(t_losses),np.mean(v_losses)
    history['train'].append(float(tm)); history['val'].append(float(vm))
    sch.step()
    print(f'Ep {epoch+1:3d} | train {tm:.4f} | val {vm:.4f}')
    # Save every 5 epochs
    if (epoch+1)%5==0:
        torch.save({'epoch':epoch,'model_state':model.state_dict(),'history':history},
                    CKPT_DIR/f'ckpt_ep{epoch+1:03d}.pth')
    # Save best
    if vm<best_val:
        best_val=vm
        torch.save({'model_state':model.state_dict(),'history':history},
                    CKPT_DIR/'best_model.pth')
        print(f'  ✓ best saved (val={best_val:.4f})')

print('Training done!')

## 8 · Validate & Visualise

In [ ]:
# Training curves
fig,ax=plt.subplots(figsize=(10,4))
ax.plot(history['train'],'b-o',ms=4,label='Train Loss')
ax.plot(history['val'],  'r-o',ms=4,label='Val Loss')
ax.set_xlabel('Epoch'); ax.set_ylabel('L1 Loss')
ax.set_title('Saree Try-On Training'); ax.legend(); ax.grid(True)
plt.tight_layout(); plt.savefig('/kaggle/working/curves.png',dpi=90); plt.show()

# Sample outputs
model.eval()
b=next(iter(vl_dl)); b={k:v.to(DEVICE) for k,v in b.items()}
with torch.no_grad():
    out=model(b['person'],b['saree'],b['blouse'],b['pose'],b['seg'],b['fabric_idx'],b['style_idx'])

def t2img(t): return ((t[0].cpu().permute(1,2,0).numpy()+1)/2*255).clip(0,255).astype(np.uint8)
titles=['Person','Saree','Blouse','Warped Saree','Warped Blouse','Rendered','Alpha']
tensors=[b['person'],b['saree'],b['blouse'],out['warped_saree'],out['warped_blouse'],out['rendered'],out['alpha_mask'].repeat(1,3,1,1)*2-1]
fig,axes=plt.subplots(1,7,figsize=(24,5))
for ax,ttl,t in zip(axes,titles,tensors):
    ax.imshow(t2img(t)); ax.set_title(ttl,fontsize=9); ax.axis('off')
plt.tight_layout(); plt.savefig('/kaggle/working/sample_output.png',dpi=90); plt.show()

## 9 · Export & Download

In [ ]:
import shutil

EXPORT = Path('/kaggle/working/saree_tryon_weights')
EXPORT.mkdir(exist_ok=True)

# Load best model
best=torch.load(str(CKPT_DIR/'best_model.pth'), map_location='cpu', weights_only=True)

# Save final weights file (matches expected path in backend)
torch.save({'model_state': best['model_state'],
             'history':     best.get('history',{})},
            EXPORT/'saree_tryon_model.pth')

# Save config
cfg = {'img_h':IMG_H,'img_w':IMG_W,'grid_size':5,'base_ch':64,
        'normalize_mean':[0.5,0.5,0.5],'normalize_std':[0.5,0.5,0.5],
        'fabric_types':['silk','cotton','georgette','chiffon','other'],
        'draping_styles':['nivi','bridal','hanging','gujarati'],
        'trained_epochs':N_EPOCHS,'best_val_loss':best_val}
with open(EXPORT/'model_config.json','w') as f: json.dump(cfg,f,indent=2)

# Also copy training curves + samples
shutil.copy('/kaggle/working/curves.png',        EXPORT/'training_curves.png')
shutil.copy('/kaggle/working/sample_output.png', EXPORT/'sample_output.png')

# Zip for download
shutil.make_archive('/kaggle/working/saree_tryon_weights','zip', EXPORT)
sz=os.path.getsize('/kaggle/working/saree_tryon_weights.zip')/1e6

print(f'\n✅  Export complete!')
print(f'   ZIP: /kaggle/working/saree_tryon_weights.zip  ({sz:.0f} MB)')
print(f'\n📥  Download steps:')
print(f'   1. Kaggle sidebar → Output → saree_tryon_weights.zip → Download')
print(f'   2. Unzip → copy saree_tryon_model.pth + model_config.json')
print(f'   3. Paste into:  sareedrap/backend/dataset/trained_models/')
print(f'   4. Flask auto-detects weights on next /api/process request')